# Phase 2: Advanced Feature Engineering
## Student Graduation Prediction System - Research Project

**Objective**: Create, engineer, dan select optimal features untuk meningkatkan model performance dan interpretability.

**Methods Covered**:
- Polynomial & polynomial interaction features
- Domain-specific ratio features
- Statistical feature selection
- Feature scaling & normalization
- Feature importance ranking (multiple methods)
- Cross-validation for feature evaluation

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('data_sample.csv')
print("[OK] Data loaded:")
print(f"  - Shape: {df.shape}")
print(f"  - Columns: {list(df.columns)}")

# Separate features & target
X = df[['ipk', 'absen', 'kegiatan']].copy()
y = df['lulus_tepat_waktu'].copy()

# Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\n[OK] Features & target separated:")
print(f"  - X shape: {X.shape}")
print(f"  - y shape: {y.shape}")
print(f"  - Target encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

## 2. Feature Engineering Pipeline

In [ ]:
print("=" * 70)
print("FEATURE ENGINEERING: CREATING NEW FEATURES")
print("=" * 70)

X_engineered = X.copy()

# ===== RATIO FEATURES (Domain Knowledge) =====
print("\n📊 1. Domain-Specific Ratio Features:")
X_engineered['ipk_per_absence'] = X['ipk'] / (X['absen'] + 1)  # Avoid division by zero
X_engineered['activity_efficiency'] = X['kegiatan'] / (X['absen'] + 1)
X_engineered['ipk_activity_ratio'] = X['ipk'] * X['kegiatan'] / 10
print("   ✓ Created 3 domain-specific ratio features")
print(f"     - ipk_per_absence: IPK per absence unit")
print(f"     - activity_efficiency: Activity participation per absence")
print(f"     - ipk_activity_ratio: Combined IPK-Activity score")

# ===== POLYNOMIAL FEATURES (Capture Non-linearity) =====
print("\n📊 2. Polynomial Features (degree=2):")
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
poly_feature_names = poly.get_feature_names_out(['ipk', 'absen', 'kegiatan'])
X_poly_df = pd.DataFrame(X_poly, columns=poly_feature_names)

# Add polynomial features (exclude originals, already in X_engineered)
for col in poly_feature_names:
    if col not in X_engineered.columns and '^' in col:
        X_engineered[col] = X_poly_df[col]

print(f"   ✓ Created {len([c for c in poly_feature_names if '^' in c])} polynomial interaction features")
print(f"     Features: {[c for c in poly_feature_names if '^' in c]}")

# ===== STATISTICAL FEATURES =====
print("\n📊 3. Statistical/Behavioral Features:")
X_engineered['attendance_risk'] = X['absen'].apply(lambda x: 1 if x > 10 else (2 if x > 5 else 3))  # 3=low risk, 1=high risk
X_engineered['ipk_category'] = pd.cut(X['ipk'], bins=[0, 3.0, 3.5, 4.0], labels=[1, 2, 3], ordered=True).astype(int)
X_engineered['activity_level'] = pd.cut(X['kegiatan'], bins=[-1, 2, 5, 10], labels=[1, 2, 3], ordered=True).astype(int)
print("   ✓ Created 3 categorical/ordinal features")

print(f"\n📈 Feature Engineering Summary:")
print(f"   Original features: {X.shape[1]}")
print(f"   Engineered features: {X_engineered.shape[1]}")
print(f"   Total features before scaling: {X_engineered.shape[1]}")
print(f"\n   All features: {list(X_engineered.columns)}")

## 3. Feature Selection & Ranking

In [ ]:
print("=" * 70)
print("FEATURE SELECTION: RANKING FEATURES BY IMPORTANCE")
print("=" * 70)

# Method 1: Univariate Statistical Test (ANOVA F-statistic)
print("\n📊 Method 1: ANOVA F-statistic (Univariate)")
selector_f = SelectKBest(score_func=f_classif, k='all')
X_sel_f = selector_f.fit_transform(X_engineered, y_encoded)
f_scores = pd.DataFrame({
    'Feature': X_engineered.columns,
    'F-Score': selector_f.scores_
}).sort_values('F-Score', ascending=False)
print(f_scores.head(10).to_string(index=False))

# Method 2: Mutual Information
print("\n📊 Method 2: Mutual Information Score")
selector_mi = SelectKBest(score_func=mutual_info_classif, k='all')
X_sel_mi = selector_mi.fit_transform(X_engineered, y_encoded)
mi_scores = pd.DataFrame({
    'Feature': X_engineered.columns,
    'MI-Score': selector_mi.scores_
}).sort_values('MI-Score', ascending=False)
print(mi_scores.head(10).to_string(index=False))

# Method 3: Permutation Importance (Model-based)
print("\n📊 Method 3: Permutation Importance (Random Forest baseline)")
rf_baseline = RandomForestClassifier(n_estimators=50, random_state=42)
rf_baseline.fit(X_engineered, y_encoded)

perm_importance = permutation_importance(rf_baseline, X_engineered, y_encoded, n_repeats=10, random_state=42)
perm_scores = pd.DataFrame({
    'Feature': X_engineered.columns,
    'Importance': perm_importance.importances_mean
}).sort_values('Importance', ascending=False)
print(perm_scores.head(10).to_string(index=False))

# Method 4: Feature Correlation with Target
print("\n📊 Method 4: Correlation with Target")
X_engineered_with_target = X_engineered.copy()
X_engineered_with_target['target'] = y_encoded
corr_target = X_engineered_with_target.corr()['target'].drop('target').abs().sort_values(ascending=False)
correlation_scores = pd.DataFrame({
    'Feature': corr_target.index,
    'Abs_Correlation': corr_target.values
})
print(correlation_scores.head(10).to_string(index=False))

# Summary ranking
print("\n" + "=" * 70)
print("OVERALL FEATURE RANKING (Consensus)")
print("=" * 70)
overall_ranking = pd.DataFrame({
    'Feature': X_engineered.columns,
    'F-Score_Rank': f_scores.reset_index(drop=True).reset_index().set_index('Feature')['index'] + 1,
    'MI_Rank': mi_scores.reset_index(drop=True).reset_index().set_index('Feature')['index'] + 1,
    'Perm_Rank': perm_scores.reset_index(drop=True).reset_index().set_index('Feature')['index'] + 1,
    'Corr_Rank': correlation_scores.reset_index(drop=True).set_index('Feature').index.map(lambda x: list(correlation_scores['Feature']).index(x) + 1)
})
overall_ranking['Average_Rank'] = overall_ranking[['F-Score_Rank', 'MI_Rank', 'Perm_Rank', 'Corr_Rank']].mean(axis=1)
overall_ranking = overall_ranking.sort_values('Average_Rank')
print(overall_ranking.to_string(index=False))

# Recommendation
print("\n✅ Recommended Features (Top 7):")
top_features = overall_ranking['Feature'].head(7).tolist()
for i, feat in enumerate(top_features, 1):
    rank = overall_ranking[overall_ranking['Feature'] == feat]['Average_Rank'].values[0]
    print(f"   {i}. {feat:25s} (Avg Rank: {rank:.1f})")

## 4. Feature Scaling & Normalization

In [ ]:
print("=" * 70)
print("FEATURE SCALING & NORMALIZATION")
print("=" * 70)

# Standardization (Z-score normalization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_engineered)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_engineered.columns)

print("\n📊 Standardization (Z-score):")
print("   Before scaling:")
print(f"     Mean: {X_engineered.mean().mean():.4f}")
print(f"     Std: {X_engineered.std().mean():.4f}")
print("   After scaling:")
print(f"     Mean: {X_scaled_df.mean().mean():.6f}")
print(f"     Std: {X_scaled_df.std().mean():.4f}")

print("\n✓ Scaling applied to all engineered features")
print(f"  - Original shape: {X_engineered.shape}")
print(f"  - Scaled shape: {X_scaled_df.shape}")

## 5. Feature Selection Strategy & Final Dataset

In [ ]:
print("=" * 70)
print("FINAL FEATURE SELECTION")
print("=" * 70)

# Strategy 1: Top-7 Features by Consensus Ranking
print("\n🎯 Strategy 1: Top-7 Features (Consensus Ranking)")
X_top7 = X_scaled_df[top_features]
print(f"   Selected {len(top_features)} features:")
for feat in top_features:
    print(f"     - {feat}")

# Strategy 2: Original Features + Best Engineered (for comparison)
print("\n🎯 Strategy 2: Original Features + Best 3 Engineered")
best_engineered = overall_ranking[overall_ranking['Feature'].isin(X_engineered.columns) & 
                                  ~overall_ranking['Feature'].isin(['ipk', 'absen', 'kegiatan'])]['Feature'].head(3).tolist()
strategy2_features = ['ipk', 'absen', 'kegiatan'] + best_engineered
X_strategy2 = X_scaled_df[strategy2_features]
print(f"   Selected {len(strategy2_features)} features:")
for feat in strategy2_features:
    print(f"     - {feat}")

# Strategy 3: All Engineered (Most complex)
print("\n🎯 Strategy 3: All Engineered Features")
X_all_engineered = X_scaled_df
print(f"   Selected {X_all_engineered.shape[1]} features (all engineered)")

# Save datasets for modeling phase
print("\n💾 Saving Feature Engineering Outputs...")
X_engineered.to_csv('features_engineered_unscaled.csv', index=False)
X_scaled_df.to_csv('features_engineered_scaled.csv', index=False)
X_top7.to_csv('features_top7_scaled.csv', index=False)
X_strategy2.to_csv('features_strategy2_scaled.csv', index=False)

print("   ✓ features_engineered_unscaled.csv")
print("   ✓ features_engineered_scaled.csv")
print("   ✓ features_top7_scaled.csv")
print("   ✓ features_strategy2_scaled.csv")

# Store recommendations
with open('feature_engineering_summary.txt', 'w') as f:
    f.write("FEATURE ENGINEERING SUMMARY\n\n")
    f.write(f"Total Original Features: {X.shape[1]}\n")
    f.write(f"Total Engineered Features: {X_engineered.shape[1]}\n\n")
    f.write("RECOMMENDED FEATURES (Top-7):\n")
    for i, feat in enumerate(top_features, 1):
        f.write(f"{i}. {feat}\n")
    f.write("\n\nFEATURE ENGINEERING TECHNIQUES APPLIED:\n")
    f.write("1. Ratio Features: ipk_per_absence, activity_efficiency, ipk_activity_ratio\n")
    f.write("2. Polynomial Features: ipk^2, absen^2, kegiatan^2, ipk×absen, ipk×kegiatan, absen×kegiatan\n")
    f.write("3. Categorical Features: attendance_risk, ipk_category, activity_level\n")
    f.write("4. Feature Scaling: StandardScaler (Z-score normalization)\n")

print("   ✓ feature_engineering_summary.txt")